# 🎯 Unidad 3 – Sesión 4
## Laboratorio: Construcción y validación del dataset analítico final
**Curso:** Minería de Datos · **Fase CRISP-DM:** Preparación de los Datos

Ruta: Diagnosticar → Limpiar → Transformar → Crear/seleccionar variables → Validar → Dataset final → Defender decisiones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_columns", 100)

data = {
    "cliente_id": [101,102,103,104,105,106,107,108,108,109,110,111],
    "edad": ["25","32","40","ERROR","29","36","150","24","24","41","33",None],
    "ciudad": ["Lima","LIMA","Cusco","CUSCO ","Arequipa","Lima","lima","Piura","Piura","Cusco",None,"AREQUIPA"],
    "fecha_registro": ["2025-01-10","2025-03-15","2024-08-01","fecha_mala","2026-01-20","2024-12-05","2025-07-17","2026-03-10","2026-03-10","2023-11-01","2025-05-30","2024-02-14"],
    "ingreso_mensual": [1500,3200,4200,2100,1800,3800,2500,1600,1600,5200,2900,np.nan],
    "gasto_mensual": [900,1800,2600,1200,1000,2500,1700,850,850,8000,1500,1900],
    "activo": ["SI","NO","SI","SI","NO","SI","Sí","si","si","NO","NO","SI"],
    "nivel_satisfaccion": ["ALTO","MEDIO","ALTO","BAJO","MEDIO","alto","BAJO","MEDIO","MEDIO","BAJO","ALTO","MEDIO"]
}
df = pd.DataFrame(data)
df_original = df.copy()
display(df)
print("Dimensiones:", df.shape)
print(df.isnull().sum())

In [ ]:
df_trabajo = df.copy()
# edad: texto -> numérico, inválidos (<18, >100) -> NaN -> mediana
df_trabajo["edad"] = pd.to_numeric(df_trabajo["edad"], errors="coerce")
df_trabajo.loc[(df_trabajo["edad"] < 18) | (df_trabajo["edad"] > 100), "edad"] = np.nan
df_trabajo["edad"] = df_trabajo["edad"].fillna(df_trabajo["edad"].median())
# categorías
for col in ["ciudad","activo","nivel_satisfaccion"]:
    df_trabajo[col] = df_trabajo[col].astype("string").str.strip().str.upper()
df_trabajo["activo"] = df_trabajo["activo"].replace({"SÍ":"SI"})
# fechas y duplicados
df_trabajo["fecha_registro"] = pd.to_datetime(df_trabajo["fecha_registro"], errors="coerce")
df_trabajo = df_trabajo.drop_duplicates(subset=["cliente_id"], keep="first").copy()
# feature engineering
ref = pd.Timestamp("2026-11-08")
df_trabajo["antiguedad_dias"] = (ref - df_trabajo["fecha_registro"]).dt.days
df_trabajo["porcentaje_gasto"] = df_trabajo["gasto_mensual"] / df_trabajo["ingreso_mensual"] * 100
df_trabajo["grupo_edad"] = pd.cut(df_trabajo["edad"], bins=[17,25,35,50,100], labels=["18-25","26-35","36-50","51+"])
# codificación
df_trabajo["activo_num"] = df_trabajo["activo"].map({"NO":0,"SI":1})
df_trabajo["satisfaccion_num"] = df_trabajo["nivel_satisfaccion"].map({"BAJO":1,"MEDIO":2,"ALTO":3})
df_trabajo = pd.concat([df_trabajo, pd.get_dummies(df_trabajo["ciudad"], prefix="ciudad", dtype=int)], axis=1)
cols_ciudad = [c for c in df_trabajo.columns if c.startswith("ciudad_")]
vars_finales = ["edad","ingreso_mensual","gasto_mensual","antiguedad_dias","porcentaje_gasto","activo_num","satisfaccion_num"] + cols_ciudad
df_analitico = df_trabajo[vars_finales].copy()
display(df_analitico)
print("Nulos restantes:\n", df_analitico.isnull().sum()[lambda s: s>0])

## Respuestas de los retos

**Reto 1 – Detective**
- Completitud 1: `edad` nula (cliente 111). Completitud 2: `ingreso_mensual` nulo (111) y `ciudad` nula (110).
- Unicidad: `cliente_id` 108 repetido (fila idéntica).
- Consistencia 1: `ciudad` (Lima/LIMA/lima, "CUSCO " con espacio). Consistencia 2: `activo` (SI/si/Sí) y `nivel_satisfaccion` (alto/ALTO).
- Validez: `edad`="ERROR", `fecha_registro`="fecha_mala", edad=150.
- Outlier: `gasto_mensual`=8000 (cliente 109, gasta 154% de su ingreso). Se investiga, no se elimina automáticamente.

**Reto 2** – `porcentaje_gasto` normaliza el gasto por el ingreso: permite comparar clientes con ingresos distintos (1500 y 5200) por el esfuerzo relativo, no por montos absolutos. Además hace evidente el caso 109 (>100%).

**Reto 3** – `LIMA=1, CUSCO=2, AREQUIPA=3` impone un orden y distancias falsas (Arequipa "mayor" que Lima, Cusco "a mitad"). Ciudad es nominal: usar One-Hot.

**Pregunta crítica** – 0 nulos no es preparación perfecta: aquí quedan 3 columnas con nulos (justificados: no imputamos ingreso ni fecha inválida), la imputación con mediana (edad) introduce valores artificiales, el cliente 110 tiene ciudad vacía (todas las dummies en 0), y un dataset limpio puede seguir sin representatividad, con fuga de información o sin sentido de negocio.

In [ ]:
# Sección 2: Quiz interactivo
import ipywidgets as widgets
from IPython.display import clear_output
preguntas = [
    ("Si edad contiene 'ERROR', ¿qué haces primero?", ["Eliminar toda la columna","Convertir con to_numeric(errors='coerce')","Reemplazar por 0","Ignorarlo"], 1, "Primero hacemos explícito el problema y luego decidimos."),
    ("Si Lima/LIMA/lima significan lo mismo, ¿qué corresponde?", ["One-Hot de inmediato","Eliminar Lima","Estandarizar texto","Cambiar todo a Cusco"], 2, "Primero corregimos consistencia."),
    ("¿cliente_id repetido es automáticamente duplicado?", ["Sí","No, depende de la granularidad","Solo si es texto","Solo si tiene nulos"], 1, "La granularidad manda."),
    ("Un outlier detectado por IQR...", ["Se elimina","Es fraude","Se investiga","Se vuelve cero"], 2, "Outlier no significa error.")
]
idx = widgets.IntSlider(value=0, min=0, max=len(preguntas)-1, description='Pregunta:')
radio = widgets.RadioButtons(); btn = widgets.Button(description='Comprobar', button_style='primary'); outq = widgets.Output()
def cargar(change=None):
    q, opts, ans, exp = preguntas[idx.value]
    radio.options = [(o, i) for i, o in enumerate(opts)]; radio.value = None
    with outq: clear_output(); print(q)
def comprobar(_):
    q, opts, ans, exp = preguntas[idx.value]
    with outq:
        clear_output(); print(q)
        if radio.value is None: print('⚠️ Selecciona una alternativa.')
        elif radio.value == ans: print('✅ Correcto.'); print(exp)
        else: print('❌ Revisa tu decisión.'); print(exp)
idx.observe(cargar, names='value'); btn.on_click(comprobar); cargar()
display(widgets.VBox([idx, radio, btn, outq]))

In [ ]:
# Sección 5: Explorador interactivo de outliers
selector = widgets.Dropdown(options=["edad","ingreso_mensual","gasto_mensual","porcentaje_gasto"], description="Variable:")
boton = widgets.Button(description="Analizar", button_style="info"); salida = widgets.Output()
def analizar(_):
    with salida:
        clear_output()
        col = selector.value; s = df_trabajo[col].dropna()
        q1, q3 = s.quantile(.25), s.quantile(.75); iqr = q3-q1; li, ls = q1-1.5*iqr, q3+1.5*iqr
        outs = df_trabajo[(df_trabajo[col]<li)|(df_trabajo[col]>ls)]
        print(f"Q1={q1:.2f} | Q3={q3:.2f} | IQR={iqr:.2f}"); print(f"Límites: {li:.2f} a {ls:.2f}")
        display(outs[["cliente_id", col]])
        plt.figure(figsize=(8,3)); plt.boxplot(s, vert=False); plt.title(f"Boxplot - {col}"); plt.show()
boton.on_click(analizar)
display(widgets.VBox([selector, boton, salida]))

In [ ]:
# Sección 9: Semáforo interactivo de preparación
criterios = ["Granularidad definida","Nulos tratados","Duplicados revisados","Categorías consistentes","Tipos correctos","Valores inválidos tratados","Outliers evaluados","Variables seleccionadas","Variables nuevas justificadas","Codificación aplicada","Dataset final validado"]
controles = {}
for criterio in criterios:
    controles[criterio] = widgets.ToggleButtons(options=["🔴","🟡","🟢"], value="🟢", description=criterio, style={"description_width":"250px"})
    display(controles[criterio])
def resumen_semaforo():
    conteo = {"🔴":0,"🟡":0,"🟢":0}
    for c in controles.values(): conteo[c.value] += 1
    print(conteo)
    if conteo["🔴"]: print("❌ Aún existen aspectos críticos.")
    elif conteo["🟡"]: print("⚠️ Está avanzado, pero requiere revisión.")
    else: print("✅ Cumple el checklist de esta práctica.")
boton_sem = widgets.Button(description="Ver resumen", button_style="success"); out_sem = widgets.Output()
def ver_sem(_):
    with out_sem: clear_output(); resumen_semaforo()
boton_sem.on_click(ver_sem); display(boton_sem, out_sem)

In [ ]:
# Sección 5: outliers por IQR (versión estática, equivale al explorador interactivo)
def outliers_iqr(d, col):
    s = d[col].dropna(); q1, q3 = s.quantile(.25), s.quantile(.75); iqr = q3-q1
    li, ls = q1-1.5*iqr, q3+1.5*iqr
    return li, ls, d[(d[col]<li)|(d[col]>ls)]
for col in ["edad","ingreso_mensual","gasto_mensual","porcentaje_gasto"]:
    li, ls, o = outliers_iqr(df_trabajo, col)
    print(f"{col}: límites {li:.1f} a {ls:.1f} | outliers -> {o['cliente_id'].tolist()}")

# Sección 7: validación automática
def validar_dataset(d):
    checks = [("No hay columnas totalmente nulas", d.isnull().all().sum()==0),
              ("No hay duplicados exactos", d.duplicated().sum()==0),
              ("Todas las columnas son numéricas", all(pd.api.types.is_numeric_dtype(d[c]) for c in d.columns)),
              ("Hay registros", len(d)>0),
              ("No hay infinitos", not np.isinf(d.select_dtypes(include=np.number)).any().any())]
    for n, ok in checks: print("✅" if ok else "❌", n)
    return checks
checks = validar_dataset(df_analitico)

# Sección 8: antes vs después
display(pd.DataFrame({
  "Indicador":["Filas","Columnas","Nulos totales","Duplicados exactos","Columnas numéricas"],
  "Original":[len(df_original),df_original.shape[1],int(df_original.isnull().sum().sum()),int(df_original.duplicated().sum()),len(df_original.select_dtypes(include=np.number).columns)],
  "Final":[len(df_analitico),df_analitico.shape[1],int(df_analitico.isnull().sum().sum()),int(df_analitico.duplicated().sum()),len(df_analitico.select_dtypes(include=np.number).columns)]}))

# 10. DESAFÍO FINAL – Dataset de tu proyecto (IBM AML: transacciones, `Is Laundering`)
Se usa una **muestra aleatoria de 100 000 filas** (semilla 42) de `/content/LI-Small_Trans.csv` (6.9 M filas, 650 MB). La celda siguiente genera `muestra_trans.csv`.

In [ ]:
import random
random.seed(42)
N = 6924049  # filas de datos (sin cabecera)
keep = set(random.sample(range(1, N+1), 100000))
muestra = pd.read_csv("/content/LI-Small_Trans.csv", dtype=str, skiprows=lambda i: i != 0 and i not in keep)
muestra.to_csv("muestra_trans.csv", index=False)
print(muestra.shape, muestra["Is Laundering"].value_counts().to_dict())

In [ ]:
# Misión 1 – Diagnóstico (evidencia real)
df_proyecto = pd.read_csv("muestra_trans.csv", dtype=str)
df_proyecto.columns = ["timestamp","banco_origen","cuenta_origen","banco_destino","cuenta_destino",
  "monto_recibido","moneda_recibida","monto_pagado","moneda_pago","formato_pago","es_lavado"]
d = df_proyecto.copy()
print("Dim:", d.shape, "| nulos totales:", int(d.isnull().sum().sum()), "| duplicados exactos:", int(d.duplicated().sum()))
for c in ["monto_recibido","monto_pagado","es_lavado"]: d[c] = pd.to_numeric(d[c], errors="coerce")
d["timestamp"] = pd.to_datetime(d["timestamp"], format="%Y/%m/%d %H:%M", errors="coerce")
print("Inválidos tras conversión:", d[["timestamp","monto_recibido","monto_pagado","es_lavado"]].isnull().sum().to_dict())
print("Fechas:", d.timestamp.min(), "->", d.timestamp.max())
print("Monedas:", d.moneda_pago.nunique(), "| formatos:", d.formato_pago.value_counts().to_dict())
n_pos = int(d.es_lavado.sum()); pct_pos = d.es_lavado.mean()
print("Clase es_lavado:", d.es_lavado.value_counts().to_dict(), f"({pct_pos:.3%} positivos)")
pct_misma = (d.moneda_recibida==d.moneda_pago).mean()
print("monto<=0:", int((d.monto_pagado<=0).sum()), "| misma moneda recibida=pagada:", f"{pct_misma:.1%}")
s = d.monto_pagado; q1, q3 = s.quantile(.25), s.quantile(.75); ls = q3+1.5*(q3-q1)
n_out = int((s>ls).sum()); pct_lav_out = d.loc[s>ls,'es_lavado'].mean()
print(f"Outliers IQR monto_pagado: {n_out} (límite {ls:,.0f}) | máx {s.max():,.0f} | p99 {s.quantile(.99):,.0f}")
print(f"% lavado entre outliers: {pct_lav_out:.3%} vs global {pct_pos:.3%}")

diagnostico = pd.DataFrame({
 "Aspecto":["Granularidad","Nulos","Duplicados","Consistencia","Validez","Outliers","Representatividad"],
 "Hallazgo":["1 fila = 1 transacción (no hay id propio)","Sin nulos","Sin duplicados exactos",
   f"Monedas/formatos ya estandarizados; {pct_misma:.1%} con misma moneda recibida/pagada","Fechas y montos convierten sin errores; sin montos <=0",
   f"{n_out/len(d):.0%} de filas sobre el límite IQR; máx {s.max():.1e}",
   f"Solo {pct_pos:.3%} positivos ({n_pos} de {len(d):,}); muestra de ~15 días"],
 "Evidencia":["Columnas: origen/destino, montos, formato, etiqueta","isnull().sum()=0","duplicated().sum()=0",
   "value_counts moneda/formato","errors='coerce' → 0 NaN",f"% lavado en outliers ≈ global ({pct_lav_out:.3%} vs {pct_pos:.3%})","value_counts es_lavado"],
 "Decision":["Mantener nivel transacción","Sin imputación","No eliminar","Sin cambio; agrupar monedas raras","Convertir tipos",
   "NO eliminar (montos altos son normales en finanzas); usar log","Reportar desbalance; usar muestra aleatoria sin filtrar por clase"]})
diagnostico

In [ ]:
# Misión 2-4: preparación, feature engineering y selección final
p = d.drop_duplicates().copy()
p["hora"] = p.timestamp.dt.hour                                        # temporal
p["dia_semana"] = p.timestamp.dt.dayofweek                             # temporal
p["es_fin_semana"] = (p.dia_semana>=5).astype(int)
p["mismo_banco"] = (p.banco_origen==p.banco_destino).astype(int)       # derivada binaria
p["misma_cuenta"] = (p.cuenta_origen==p.cuenta_destino).astype(int)
p["misma_moneda"] = (p.moneda_recibida==p.moneda_pago).astype(int)
p["log_monto"] = np.log1p(p.monto_pagado)                              # numérica derivada
p["rango_monto"] = pd.qcut(p.monto_pagado, 4, labels=["Q1_bajo","Q2","Q3","Q4_alto"])  # categórica discretizada
top = p.moneda_pago.value_counts().head(5).index
p["moneda_grupo"] = p.moneda_pago.where(p.moneda_pago.isin(top), "Otra")
dup_logico = int(p.duplicated().sum())

df_final = pd.concat([
  p[["log_monto","hora","dia_semana","es_fin_semana","mismo_banco","misma_cuenta","misma_moneda","es_lavado"]],
  pd.get_dummies(p.formato_pago, prefix="fmt", dtype=int),
  pd.get_dummies(p.moneda_grupo, prefix="mon", dtype=int),
  pd.get_dummies(p.rango_monto, prefix="rng", dtype=int)], axis=1)
display(df_final.head())
print("Dim final:", df_final.shape, "| nulos:", int(df_final.isnull().sum().sum()))
print("Filas idénticas en df_final:", int(df_final.duplicated().sum()), "(esperado: sin id, transacciones distintas pueden compartir features; originales sin duplicados:", dup_logico, ")")
for c in ["misma_cuenta","mismo_banco","es_fin_semana"]:
    print(c, df_final.groupby(c).es_lavado.mean().round(5).to_dict())
# validación (el check de duplicados exactos se evalúa sobre filas ya sin id: ver nota)
_ = validar_dataset(df_final)

plan_preparacion = pd.DataFrame({
 "Variable":["timestamp","monto_pagado","banco/cuenta origen y destino","moneda_pago","formato_pago","monto_recibido / moneda_recibida"],
 "Problema_o_necesidad":["Texto, no usable","Muy sesgado (máx 1e11)","IDs de alta cardinalidad","15 categorías, varias raras","7 categorías nominales","Redundante con monto_pagado en 98.6%"],
 "Decision":["Convertir a datetime","Log + discretizar","No usar como feature; derivar comparaciones","Agrupar top 5 + 'Otra'","One-Hot","Descartar; conservar misma_moneda"],
 "Transformacion":["to_datetime → hora, dia_semana, es_fin_semana","np.log1p, qcut 4","mismo_banco, misma_cuenta","where + get_dummies","get_dummies","misma_moneda"],
 "Justificacion":["Patrones horarios suelen indicar anomalías","Reduce asimetría sin borrar casos reales","Evita memorizar IDs y fuga","Evita columnas casi vacías","Sin orden natural; no codificar 1..7","Evita multicolinealidad"],
 "Validacion":["0 NaT","Sin infinitos","Sin nulos","Suma de dummies = 1","Suma de dummies = 1","validar_dataset"]})
seleccion_final = pd.DataFrame({
 "Variable":["es_lavado","log_monto","hora","dia_semana / es_fin_semana","mismo_banco / misma_cuenta","misma_moneda","fmt_* / mon_* / rng_*","cuentas y bancos (IDs)","monto_recibido, moneda_recibida","timestamp"],
 "Rol":["Objetivo","Predictora","Predictora","Predictora","Predictora","Predictora","Predictora (codificadas)","Identificador","Redundante","Fuente de derivadas"],
 "Decision":["Mantener","Mantener","Mantener","Mantener","Mantener","Mantener","Mantener","Excluir","Excluir","Excluir (reemplazada)"],
 "Motivo":["Etiqueta del proyecto","Monto normalizado","Patrón temporal","Patrón semanal","Comportamiento de la relación origen-destino","Conversión de divisa","Categóricas sin orden","Cardinalidad alta, riesgo de sobreajuste","Casi idéntica a monto_pagado","Ya descompuesta en hora/día"]})
display(plan_preparacion); display(seleccion_final)

## Misión 5 – Checklist y conclusión final

1. Granularidad: 1 fila = 1 transacción ✅  2. Tipos: fecha y montos convertidos ✅  3. Nulos: 0 ✅  4. Duplicados originales: 0 ✅ (las 286 filas idénticas en el final son esperables al excluir IDs)  5. Categorías estandarizadas ✅  6. Inválidos: 0 tras conversión ✅  7. Outliers evaluados: no se eliminan, se usa `log_monto` ✅  8. Variables alineadas al objetivo (detectar lavado) ✅  9. Derivadas interpretables ✅  10. Proceso reproducible (muestra con semilla 42) ✅

**Conclusión (≤8 líneas):** El dataset es consistente (sin nulos, duplicados ni tipos inválidos), por lo que la preparación se centró en transformar y crear variables: 25 columnas numéricas con la fecha descompuesta, `log_monto`, rangos de monto, indicadores de mismo banco/cuenta/moneda y One-Hot de formato y moneda. Los IDs se excluyeron por su alta cardinalidad. Limitaciones: solo 0.063% de casos positivos (63 de 100 000; desbalance severo, usar estratificación y métricas como recall o AUC-PR), muestra de 100 000 filas de ~15 días, y los outliers de monto se conservaron porque su tasa de lavado es igual a la global. Siguiente paso: dividir train/test estratificado antes de escalar o balancear, para evitar fuga de información.